# Lesson 07 - പ്ലാനിംഗ് ഡിസൈൻ പാറ്റേൺ

ഈ നോട്ട്ബുക്ക് മൈക്രൊസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിംവർക്ക് ഉപയോഗിച്ച് AI ഏജന്റുകൾക്കായുള്ള **പ്ലാനിംഗ് ഡിസൈൻ പാറ്റേൺ** പ്രദർശിപ്പിക്കുന്നു.  
നിങ്ങൾക്ക് സമ്പൂർണ്ണ യാത്രാ അഭ്യർത്ഥനകൾ ഘടിപ്പിച്ച്, അവയെ ഘടനാപരമായ ഉപപ്രവൃത്തികളായി വിഭജിച്ച്, വിദഗ്ധ ഏജന്റുകൾക്ക് ഏൽപ്പിച്ച്, ഫലമായി രൂപപ്പെടുന്ന പ്ലാനിനെ പ്രവർത്തിപ്പിക്കുന്നത് എങ്ങനെ ചെയ്യാമെന്ന് പഠിക്കാനാകും — എല്ലാം Pydantic മാതൃകകൾ ഉപയോഗിച്ച ഘടനാപരമായ ഔട്ട്പുട്ട് വഴി.


## സജ്ജീകരണം


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Task Decomposition

ടാസ്‌ക് ഡികംപ്പോസിഷൻ പ്ലാനിങ് ഡിസൈൻ പാറ്റേണിന്റെ മദ്ധ്യമാണ്. ഒരു സിംപിൾ ഏജന്റിനോട് മുഴുവൻ കോംപ്ലക്സായ അഭ്യർത്ഥന കൈകാര്യം ചെയ്യാൻ പറയാനുള്ള പകരം, പ്രശ്നം ചെറുതും നന്നായി നിർവചിക്കപ്പെട്ട **സബ്‌టാസ്കുകൾ** ആയി വിഭജിക്കുന്നു. ഓരോ സബ്‌‌ടാസ്കും ക്ലിയർ പ്രായോറിറ്റികളും ആശ്രിത ക്രമവും ഉള്ള സ്പെഷ്യലിസ്റ്റ് ഏജന്റിനെ (ഉദാ: ഫ്‌ളൈറ്റുകൾ, ഹോട്ടലുകൾ, ആക്റ്റിവിറ്റികൾ) ഏൽപ്പിക്കുന്നു.

ഈ സമീപനം പല സുവിദ്ധകളും നൽകുന്നു:
- **പകുതിമാറ്റം**: ഓരോ സബ്‌‌ടാസ്കിനും ഒരൊറ്റ ഉത്തരവാദിത്വം.
- **പാരലലിസം**: സ്വതന്ത്ര സബ്‌‌ടാസ്കുകൾ ഏകദേശം ഒരേസമയം പ്രവർത്തിക്കാം.
- **നിലവാരം**: പരാജയങ്ങൾ വ്യക്തിഗത സബ്‌‌ടാസ്കുകളിലേക്ക് ഒതുക്കുന്നു.
- **ബഡ്ജറ്റ് ട്രാക്കിംഗ്**: ചിലവുകൾ സബ്‌‌ടാസ്കുകൾക്ക് അനുസരിച്ച് കണക്കാക്കി ഒത്തുചേർക്കുന്നു.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## ക്രമീകരിച്ച ഔട്ട്പുട്ടോടുകൂടിയ ഒരു പ്ലാനിംഗ് ഏജന്റ് സൃഷ്ടിക്കൽ

പ്ലാനിംഗ് ഏജന്റ് ഒരു **ഫ്രണ്ട് ഡിസ്ക് കോഓർഡിനേറ്റർ** ആയി പ്രവർത്തിക്കുന്നു. ഉയർന്ന തലത്തിലുള്ള യാത്രാ അഭ്യർത്ഥന നൽകുമ്പോൾ, അത് ഒരു ക്രമീകരിച്ച `TravelPlan` രൂപപ്പെടുത്തുന്നു — അഭ്യർത്ഥന സബ്ടാസ്കുകളിൽ പിരിക്കുകയാണ്, മുൻഗണനകൾ നല്കുകയാണ്, അന اینکه്യങ്ങളെ തിരിച്ചറിയുകയാണ്, അതിനാൽ ഒരു കോൺസിയർജ് അല്ലെങ്കിൽ നിർവഹണ തല പ്രവർത്തനം നടത്തി.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## വിദഗ്ധ ഉപകരണങ്ങളോടുകൂടെ ഒരു പദ്ധതിയുടെ നടപ്പാക്കല്‍

ഫ്രണ്ട് ഡിസ്ക് ഏജന്റ് ഒരു ഘടനയിലുള്ള പദ്ധതി തയ്യാറാക്കിയ ശേഷം, **കൺസിയർജ് ഏജന്റ്** അതിനെ നടപ്പിലാക്കും. 
ഓരോ വിദഗ്ധ ഉപകരണവും ഒരു ഉപപ്രവർത്തന വിഭാഗം (ഫ്ലൈറ്റുകൾ, ഹോട്ടലുകൾ, പ്രവർത്തനങ്ങൾ) കൈകാര്യം ചെയ്യും. കൺസിയർജ് പദ്ധതിയിലെ ഉപപ്രവർത്തനങ്ങളെ ആശ്രിതക്രമത്തിൽ പിന്വറ്റി, ഓരോത്തെയും അനുയോജ്യമായ ഉപകരണത്തിലേക്ക് അയക്കും.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## സാരാംശം

ഈ പാഠത്തിൽ നിങ്ങൾ **പ്ലാനിംഗ് ഡിസൈൻ പാറ്റേൺ** AI ഏജന്റുകൾക്കായി പഠിച്ചു:

1. **ടാസ്‌ക് ഡികമ്പോസിഷൻ** — ഒരു ഫ്രണ്ട് ഡെസ്ക് പ്ലാനിംഗ് ഏജന്റ് ഒരു സങ്കീർണ്ണമായ യാത്രാ അഭ്യർത്ഥന ഏകോപിതമായ സബ്ടാസ്കുകളായി പൈഡാന്റിക് മോഡലുകൾ ഉപയോഗിച്ച് വിഭജിച്ച്, ഓരോതും ഒരു വിന്യാസജ്ഞ ഏജന്റിനും മുൻഗണനകളും ആശ്രിതത്വങ്ങളും നൽകിയിരിക്കും.
2. **സംരചിത ഔട്ട്പുട്ട്** — `response_format` പാസാക്കുമ്പോൾ ഏജന്റ് സ്വതന്ത്ര വാചകത്തിന്റെ പകരം സാധൂകരിച്ച `TravelPlan` ഓബ്‌ജക്റ്റ് തിരികെ നൽകുന്നു, ഇത് നിമിഷാനന്തര പ്രോസസ്സിംഗിന് വിശ്വാസയോഗ്യമാക്കുന്നു.
3. **പ്ലാൻ നടപ്പാക്കൽ** — കോൺസിയർജ് ഏജന്റ് സബ്ടാസ്കുകൾ ഹൈപ്പർസ്പെഷ്യലൈസ്റ്റ് ഉപകരണങ്ങൾ (`book_flight`, `reserve_hotel`, `book_activity`) ഉപയോഗിച്ച് പ്ലാൻ നടപ്പാക്കുകയും ഫലങ്ങൾ റിപ്പോർട്ട് ചെയ്യുകയും ചെയ്‍തു.

ഈ പാറ്റേൺ *എന്ത് ചെയ്യണമെന്ന്* (പ്ലാനിംഗ്) *എങ്ങനെ ചെയ്യണമെന്ന്* (നടപ്പാക്കൽ) എന്ന കാര്യങ്ങൾ വിഷമീകരിച്ച് ഏജന്റുകൾ കൂടുതൽ മൊഡ്യൂളറായ, പരീക്ഷണയോഗ്യമായ, വിപുലീകരിക്കാൻ എളുപ്പമുള്ളവയാക്കി മാറ്റുന്നു.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അസ്വീകാര്യത**:  
ഈ പ്രമാണം AI തർജ്ജമാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് തർജ്ജമ ചെയ്തതാണ്. ഞങ്ങൾ സത്യതയ്ക്ക് ശ്രമിക്കുന്നുവെങ്കിലും, സ്വയം പ്രവർത്തിക്കുന്ന തർജ്ജമകളിൽ തെറ്റുകളും കൃത്യതയിൽ കുറവുകളും ഉണ്ടാകാമെന്ന് ദയവായി മനസിലാക്കുക. പ്രമാണത്തിന്റെ ജന്മഭാഷയിലുള്ള അസൽ പകർപ്പ് മാത്രമാണ് വിശ്വസനീയമായ ഉറവിടം. പ്രധാന വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യൻ്റെ തർജ്ജമ നിർദ്ദേശിക്കപ്പെടുന്നു. ഈ തർജ്ജമ ഉപയോഗിച്ചുകൊണ്ടുണ്ടാകുന്ന ഏതെങ്കിലും തെറ്റിദ്ധാരണകൾക്കും വ്യാഖ്യാനദോഷങ്ങൾക്കും ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
